In [1]:

from pathlib import Path

# Locate the extracted VisDrone dataset.
VISDRONE_ROOT = (
    Path.home()
    / "Downloads"
    / "VisDrone2019-MOT-val"
    / "VisDrone2019-MOT-val"
)

SEQUENCES_DIR = VISDRONE_ROOT / "sequences"
ANNOTATIONS_DIR = VISDRONE_ROOT / "annotations"

print("Dataset exists:", VISDRONE_ROOT.exists())
print("Sequences folder exists:", SEQUENCES_DIR.exists())
print("Annotations folder exists:", ANNOTATIONS_DIR.exists())

if not SEQUENCES_DIR.exists() or not ANNOTATIONS_DIR.exists():
    raise FileNotFoundError(
        "Check the VISDRONE_ROOT path."
    )

# List video sequences.
sequence_dirs = sorted(
    p for p in SEQUENCES_DIR.iterdir() if p.is_dir()
)

print("\nNumber of sequences:", len(sequence_dirs))

for sequence in sequence_dirs[:10]:
    images = list(sequence.rglob("*.jpg"))
    print(sequence.name, "| Frames:", len(images))

# List annotation files.
annotation_files = sorted(
    ANNOTATIONS_DIR.rglob("*.txt")
)

print("\nNumber of annotation files:", len(annotation_files))

for annotation in annotation_files[:10]:
    print(annotation.name)

# Inspect the first annotation file.
if annotation_files:
    sample_file = annotation_files[0]

    print("\nSample annotation:", sample_file.name)

    with open(sample_file, "r", encoding="utf-8-sig") as f:
        for i, line in enumerate(f):
            if i == 5:
                break
            print(line.strip())


Dataset exists: True
Sequences folder exists: True
Annotations folder exists: True

Number of sequences: 7
uav0000086_00000_v | Frames: 464
uav0000117_02622_v | Frames: 349
uav0000137_00458_v | Frames: 233
uav0000182_00000_v | Frames: 363
uav0000268_05773_v | Frames: 978
uav0000305_00000_v | Frames: 184
uav0000339_00001_v | Frames: 275

Number of annotation files: 7
uav0000086_00000_v.txt
uav0000117_02622_v.txt
uav0000137_00458_v.txt
uav0000182_00000_v.txt
uav0000268_05773_v.txt
uav0000305_00000_v.txt
uav0000339_00001_v.txt

Sample annotation: uav0000086_00000_v.txt
102,0,38,666,71,88,1,1,1,0
103,0,45,662,71,91,1,1,1,0
104,0,52,658,72,95,1,1,1,0
105,0,60,654,71,99,1,1,1,0
106,0,67,651,72,102,1,1,1,0


# AeroTrack — Part 4: Multi-Video Global Identity Management

## Objective

Part 1 creates short-lived BoT-SORT tracker IDs. Part 3 extracts OSNet appearance embeddings from person tracklets. This notebook combines those outputs into a **provisional global identity registry** that can compare tracklets from one or more videos.

A local tracker ID is only meaningful inside its own video/tracking run. AeroTrack therefore identifies a tracklet using:

`video_id + track_id`

For example, `session_01::track_14` and `session_02::track_14` are different tracklets even though both have local tracker ID 14.

## What this notebook does

1. Load one or more source videos and their detailed tracking logs.
2. Extract representative person crops for each `(video_id, track_id)`.
3. Use pretrained OSNet to produce appearance embeddings.
4. Compare tracklets from different videos and non-overlapping tracklets from the same video.
5. Rank likely matches and calculate a best-versus-second-best margin.
6. Propose conservative matches only when appearance score, mutual-best match and margin conditions all pass.
7. Prevent a same-video cluster from containing two tracklets that overlap in time.
8. Create a saved provisional global identity registry and a human-review queue.

## Important: provisional does not mean verified

The score bands in this notebook are starting settings, not calibrated probabilities. A cosine similarity of 0.92 does **not** mean a 92% chance of identity.

High-scoring pairs are recorded as `high_similarity_proposal`, not as verified people. Moderate matches are marked `review_required`. All identities remain anonymous.

The pretrained OSNet model is trained on general pedestrian imagery, not overhead drone views. Cross-video matching may be unreliable when the cameras, altitude, lighting or viewing angle differ substantially. We need labelled examples from AeroTrack footage to calibrate thresholds before relying on automatic counts.

References:
- ReID package and OSNet API: https://reid.roboflow.com/
- Pretrained model details: https://reid.roboflow.com/guides/loading/
- Aerial person ReID challenges and datasets: https://openaccess.thecvf.com/content/WACV2024W/RWS/papers/Khaldi_Unsupervised_Person_Re-Identification_in_Aerial_Imagery_WACVW_2024_paper.pdf

## Section 1 — Project paths and configuration

Keep this notebook in the AeroTrack folder.

The first entry below uses your Experiment 008 video and tracking log. To compare another video, add another entry with a unique `video_id`, the video path, and a matching detailed tracking CSV.

**Important:** The tracking CSV must contain detection-level rows with these columns:

`frame, track_id, confidence, x1, y1, x2, y2`

It may also contain `time_seconds`. The tracking log from Experiment 008 has this format.

A summary CSV containing only one row per tracker ID is not sufficient.

In [ ]:
from pathlib import Path
import time
import math
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from IPython.display import display
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

REID_OUTPUT_DIR = OUTPUT_DIR / "part4_global_identity"
CROP_DIR = REID_OUTPUT_DIR / "tracklet_crops"

for folder in [DATA_DIR, OUTPUT_DIR, REID_OUTPUT_DIR, CROP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Add one dictionary for each video/tracking-log pair.
# Keep each video_id unique and do not reuse it for a different recording.
VIDEO_SOURCES = [
    {
        "video_id": "session_01",
        "video_path": DATA_DIR / "drone_video.mov",
        "track_log_path": OUTPUT_DIR / "continuity_track_log.csv",
        # Set to a number if known; None estimates it from time_seconds and frame.
        "segment_start_seconds": None
    },

    # Example for a second video. Uncomment and update the filenames.
    # {
    #     "video_id": "session_02",
    #     "video_path": DATA_DIR / "drone_video_02.mov",
    #     "track_log_path": OUTPUT_DIR / "drone_video_02_track_log.csv",
    #     "segment_start_seconds": 0.0
    # }
]

# OSNet and crop-selection settings.
REID_MODEL_NAME = "osnet_x1_0_msmt17_combineall"
MIN_CONFIDENCE = 0.25
MIN_BOX_WIDTH = 20
MIN_BOX_HEIGHT = 40
MIN_TRACK_OBSERVATIONS = 5
MAX_CROPS_PER_TRACKLET = 5
CROP_MARGIN_RATIO = 0.05
EMBEDDING_BATCH_SIZE = 8

# Initial candidate-ranking settings.
# These are experimental screening values, not calibrated probabilities.
AUTO_MATCH_THRESHOLD = 0.92
REVIEW_THRESHOLD = 0.78
MIN_SIMILARITY_MARGIN = 0.05
MAX_CANDIDATES_PER_TRACKLET = 5

# Same-video overlapping tracklets are never allowed to merge.
# Different videos have independent timelines, so no cross-video time gap is used.

print("Project folder:", PROJECT_DIR)
print("Configured videos:", len(VIDEO_SOURCES))
for source in VIDEO_SOURCES:
    print(
        source["video_id"],
        "| video exists:", Path(source["video_path"]).exists(),
        "| log exists:", Path(source["track_log_path"]).exists()
    )

## Section 2 — Check GPU and install/import ReID

We use the existing PyTorch installation. Do not reinstall or upgrade PyTorch in this notebook.

If the `reid` package is not installed, run the install cell, restart the kernel if Jupyter requests it, and then continue.

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    REID_DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    DEVICE = "cpu"
    REID_DEVICE = "cpu"
    print("CUDA not available. Using CPU.")

print("Detection/tracking device:", DEVICE)
print("ReID device:", REID_DEVICE)

In [ ]:
%pip install -U reid

In [ ]:
from reid import ReIDModel

print("ReID library imported successfully.")

## Section 3 — Optional: create a detailed tracking log for a new video

Use this helper only when a new video does not already have a detailed tracker log.

This runs the original pretrained YOLO26s detector, tiled inference and BoT-SORT, then writes one row per detected person per frame. It can be slow for 4K video because each frame is processed in overlapping tiles.

For a video you have already processed, use its existing detailed log instead.

When calling the function, give the video a unique `video_id`. The output log stores frame-local numbers and original-video timestamps.

In [ ]:
def create_tracking_log(
    video_path,
    output_csv,
    start_seconds=0.0,
    duration_seconds=None,
    video_id="new_session"
):
    video_path = Path(video_path)
    output_csv = Path(output_csv)

    if not video_path.exists():
        raise FileNotFoundError(video_path)

    detector = YOLO("yolo26s.pt")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 25.0

    cap.set(cv2.CAP_PROP_POS_MSEC, start_seconds * 1000)

    max_frames = (
        int(duration_seconds * fps)
        if duration_seconds is not None else None
    )

    output_csv.parent.mkdir(parents=True, exist_ok=True)

    rows = []
    frame_number = 0
    start_time = time.time()

    while max_frames is None or frame_number < max_frames:
        success, frame = cap.read()
        if not success:
            break

        frame_number += 1

        result = detector.track(
            source=frame,
            persist=True,
            tracker="botsort.yaml",
            classes=[0],
            conf=0.25,
            imgsz=640,
            tile=640,
            tile_overlap=0.20,
            max_det=300,
            device=DEVICE,
            verbose=False
        )[0]

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.int().cpu().tolist()
            scores = result.boxes.conf.cpu().tolist()

            for box, track_id, score in zip(boxes, ids, scores):
                x1, y1, x2, y2 = box.tolist()

                rows.append({
                    "video_id": video_id,
                    "frame": frame_number,
                    "time_seconds": start_seconds + frame_number / fps,
                    "track_id": int(track_id),
                    "confidence": float(score),
                    "x1": float(x1),
                    "y1": float(y1),
                    "x2": float(x2),
                    "y2": float(y2)
                })

        if frame_number % 100 == 0:
            print(
                f"{video_id}: processed {frame_number} frames; "
                f"elapsed {time.time() - start_time:.1f}s"
            )

    cap.release()

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    log_df = pd.DataFrame(rows)
    log_df.to_csv(output_csv, index=False)

    print("Tracking log saved:", output_csv)
    print("Frames processed:", frame_number)
    print("Detection observations:", len(log_df))
    print("Tracker IDs:", log_df["track_id"].nunique() if not log_df.empty else 0)

    return log_df


# Example (do not run unless you need to create a new log):
# new_log = create_tracking_log(
#     video_path=DATA_DIR / "drone_video_02.mov",
#     output_csv=OUTPUT_DIR / "drone_video_02_track_log.csv",
#     start_seconds=0,
#     duration_seconds=30,
#     video_id="session_02"
# )

## Section 4 — Load, validate and combine tracking logs

Each video has its own local tracker IDs. We create a unique `tracklet_key` using both the video ID and local tracker ID.

For example:
- `session_01::track_14`
- `session_02::track_14`

These are two different tracklets unless the ReID layer proposes a match.

In [ ]:
required_columns = {
    "frame", "track_id", "confidence", "x1", "y1", "x2", "y2"
}

all_logs = []
video_metadata = {}

if not VIDEO_SOURCES:
    raise ValueError("Add at least one video to VIDEO_SOURCES.")

video_ids = [source["video_id"] for source in VIDEO_SOURCES]
if len(video_ids) != len(set(video_ids)):
    raise ValueError("Each configured video must have a unique video_id.")

for source_order, source in enumerate(VIDEO_SOURCES):
    video_id = str(source["video_id"])
    video_path = Path(source["video_path"])
    log_path = Path(source["track_log_path"])

    if not video_path.exists():
        raise FileNotFoundError(f"{video_id}: video not found: {video_path}")
    if not log_path.exists():
        raise FileNotFoundError(f"{video_id}: tracking log not found: {log_path}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"{video_id}: could not open video.")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps <= 0:
        raise ValueError(f"{video_id}: invalid video FPS.")

    log = pd.read_csv(log_path)
    missing = required_columns - set(log.columns)
    if missing:
        raise ValueError(f"{video_id}: missing columns {missing}")

    numeric_columns = list(required_columns)
    for column in numeric_columns:
        log[column] = pd.to_numeric(log[column], errors="coerce")

    log = log.dropna(subset=numeric_columns).copy()
    log["frame"] = log["frame"].astype(int)
    log["track_id"] = log["track_id"].astype(int)

    configured_start = source.get("segment_start_seconds")

    if configured_start is not None:
        segment_start = float(configured_start)
    elif "time_seconds" in log.columns:
        log["time_seconds"] = pd.to_numeric(
            log["time_seconds"], errors="coerce"
        )
        valid_times = log.dropna(subset=["time_seconds"])
        if valid_times.empty:
            segment_start = 0.0
        else:
            offsets = (
                valid_times["time_seconds"] -
                valid_times["frame"] / fps
            )
            segment_start = float(offsets.median())
            deviation = float((offsets - segment_start).abs().max())
            if deviation > 0.10:
                raise ValueError(
                    f"{video_id}: frame/time mapping varies by "
                    f"{deviation:.3f}s. Set segment_start_seconds "
                    "explicitly and check that this log belongs to this video."
                )
    else:
        segment_start = 0.0

    # Set time_seconds if the input log did not contain it.
    if "time_seconds" not in log.columns:
        log["time_seconds"] = (
            segment_start + log["frame"] / fps
        )
    else:
        # Fill missing timestamps using the frame number and inferred segment start.
        log["time_seconds"] = log["time_seconds"].fillna(
            segment_start + log["frame"] / fps
        )

    log["video_id"] = video_id
    log["source_order"] = source_order
    log["video_path"] = str(video_path)
    log["track_log_path"] = str(log_path)
    log["fps"] = fps
    log["segment_start_seconds"] = segment_start
    log["tracklet_key"] = (
        log["video_id"] + "::track_" + log["track_id"].astype(str)
    )

    all_logs.append(log)

    video_metadata[video_id] = {
        "video_path": video_path,
        "fps": fps,
        "width": width,
        "height": height,
        "segment_start_seconds": segment_start,
        "source_order": source_order
    }

    print(
        f"{video_id}: {len(log)} observations, "
        f"{log['track_id'].nunique()} tracklets, "
        f"{width}x{height}, {fps:.2f} FPS, "
        f"segment starts at {segment_start:.2f}s"
    )

track_log = pd.concat(all_logs, ignore_index=True)

print("\nCombined observations:", len(track_log))
print("Combined tracklets:", track_log["tracklet_key"].nunique())
display(track_log.head())

## Section 5 — Select representative crops for every video/tracklet

We select up to five detections per tracklet, spread across its observed time range. Within each time segment, we prefer a crop with higher detector confidence and a larger bounding box.

The dimensions below are minimum crop dimensions in the original video. We keep them modest because overhead people may be small. Very small crops are still included in the raw tracking log but excluded from this first OSNet feature experiment.

In [ ]:
track_log["box_width"] = track_log["x2"] - track_log["x1"]
track_log["box_height"] = track_log["y2"] - track_log["y1"]
track_log["box_area"] = (
    track_log["box_width"].clip(lower=0) *
    track_log["box_height"].clip(lower=0)
)
track_log["quality_score"] = (
    track_log["confidence"] *
    np.sqrt(track_log["box_area"])
)

usable_log = track_log[
    (track_log["confidence"] >= MIN_CONFIDENCE) &
    (track_log["box_width"] >= MIN_BOX_WIDTH) &
    (track_log["box_height"] >= MIN_BOX_HEIGHT)
].copy()

usable_counts = usable_log.groupby("tracklet_key").size()
eligible_tracklets = usable_counts[
    usable_counts >= MIN_TRACK_OBSERVATIONS
].index.tolist()

usable_log = usable_log[
    usable_log["tracklet_key"].isin(eligible_tracklets)
].copy()

selected_rows = []

for tracklet_key, group in usable_log.groupby("tracklet_key"):
    group = group.sort_values("frame").reset_index(drop=True)

    number_of_bins = min(MAX_CROPS_PER_TRACKLET, len(group))
    position_bins = np.array_split(
        np.arange(len(group)),
        number_of_bins
    )

    for sample_number, positions in enumerate(position_bins, start=1):
        if len(positions) == 0:
            continue

        time_bin = group.iloc[positions]
        best_index = time_bin["quality_score"].idxmax()

        selected = time_bin.loc[best_index].to_dict()
        selected["sample_number"] = sample_number
        selected_rows.append(selected)

selected_crops = pd.DataFrame(selected_rows)

if selected_crops.empty:
    raise ValueError(
        "No tracklets passed crop selection. Lower the minimum crop size "
        "or confidence after inspecting the tracking log."
    )

print("Tracklets in raw logs:", track_log["tracklet_key"].nunique())
print("Eligible tracklets:", selected_crops["tracklet_key"].nunique())
print("Representative crops selected:", len(selected_crops))

display(
    selected_crops[
        ["video_id", "track_id", "tracklet_key", "frame",
         "confidence", "box_width", "box_height", "quality_score"]
    ].head(20)
)

## Section 6 — Extract crops from each source video

The log contains box coordinates in the original video frame. This section reads each source video, seeks to its segment start, and extracts only the selected frames.

Crops are stored in video-specific folders, so identical local tracker IDs from different videos never overwrite each other.

In [ ]:
selected_by_video = {
    video_id: {
        int(frame_number): group
        for frame_number, group in video_group.groupby("frame")
    }
    for video_id, video_group in selected_crops.groupby("video_id")
}

crop_records = []

for source in VIDEO_SOURCES:
    video_id = str(source["video_id"])
    meta = video_metadata[video_id]
    selected_frames = selected_by_video.get(video_id, {})

    if not selected_frames:
        print(f"{video_id}: no selected crops.")
        continue

    cap = cv2.VideoCapture(str(meta["video_path"]))
    cap.set(
        cv2.CAP_PROP_POS_MSEC,
        meta["segment_start_seconds"] * 1000
    )

    if not cap.isOpened():
        raise RuntimeError(f"{video_id}: could not open video for cropping.")

    max_local_frame = max(selected_frames)
    frames_read = 0

    while frames_read < max_local_frame:
        success, frame = cap.read()
        if not success:
            break

        frames_read += 1

        if frames_read not in selected_frames:
            continue

        for _, row in selected_frames[frames_read].iterrows():
            x1, y1, x2, y2 = map(
                float, [row["x1"], row["y1"], row["x2"], row["y2"]]
            )

            box_width = x2 - x1
            box_height = y2 - y1

            margin_x = int(box_width * CROP_MARGIN_RATIO)
            margin_y = int(box_height * CROP_MARGIN_RATIO)

            left = max(0, int(x1) - margin_x)
            top = max(0, int(y1) - margin_y)
            right = min(frame.shape[1], int(x2) + margin_x)
            bottom = min(frame.shape[0], int(y2) + margin_y)

            crop = frame[top:bottom, left:right]

            if crop.size == 0:
                continue

            crop_height, crop_width = crop.shape[:2]
            if crop_width < 2 or crop_height < 2:
                continue

            track_id = int(row["track_id"])
            track_folder = (
                CROP_DIR / video_id / f"track_{track_id:04d}"
            )
            track_folder.mkdir(parents=True, exist_ok=True)

            crop_path = track_folder / (
                f"frame_{frames_read:06d}_"
                f"sample_{int(row['sample_number']):02d}.jpg"
            )

            if not cv2.imwrite(str(crop_path), crop):
                continue

            crop_records.append({
                "video_id": video_id,
                "track_id": track_id,
                "tracklet_key": row["tracklet_key"],
                "frame": frames_read,
                "time_seconds": float(row["time_seconds"]),
                "confidence": float(row["confidence"]),
                "box_width": crop_width,
                "box_height": crop_height,
                "quality_score": float(row["quality_score"]),
                "crop_path": str(crop_path)
            })

    cap.release()

crop_manifest = pd.DataFrame(crop_records)

if crop_manifest.empty:
    raise RuntimeError(
        "No crops were extracted. Check that the video files and tracking "
        "logs refer to the same recordings and segment starts."
    )

CROP_MANIFEST_PATH = REID_OUTPUT_DIR / "multi_video_crop_manifest.csv"
crop_manifest.to_csv(CROP_MANIFEST_PATH, index=False)

print("Crops saved:", len(crop_manifest))
print("Tracklets with crops:", crop_manifest["tracklet_key"].nunique())
print("Crop manifest:", CROP_MANIFEST_PATH)

display(crop_manifest.head())

## Section 7 — Inspect a representative crop per tracklet

Before extracting embeddings, check that crops show the correct person. This catches incorrect video/log pairings, time-offset mistakes, poor boxes and crops that mostly contain background.

In [ ]:
representative_crops = (
    crop_manifest
    .sort_values("quality_score", ascending=False)
    .drop_duplicates("tracklet_key")
    .sort_values(["video_id", "track_id"])
    .reset_index(drop=True)
)

columns = 5
rows = math.ceil(len(representative_crops) / columns)

fig, axes = plt.subplots(
    rows, columns,
    figsize=(columns * 3, rows * 3)
)
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis("off")

for index, row in representative_crops.iterrows():
    image = cv2.imread(row["crop_path"])
    if image is None:
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    axes[index].imshow(image)
    axes[index].set_title(
        f"{row['video_id']}\n"
        f"Track {int(row['track_id'])} | "
        f"{int(row['box_width'])}x{int(row['box_height'])}"
    )
    axes[index].axis("off")

plt.tight_layout()
plt.show()

## Section 8 — Load pretrained OSNet

The model converts each person crop into a 512-dimensional appearance embedding. The checkpoint is pretrained on MSMT17, not aerial footage, so this is a baseline feature extractor.

The package's path-based extraction reads saved crops and applies its model-specific preprocessing.

In [ ]:
reid_model = ReIDModel.from_pretrained(
    REID_MODEL_NAME,
    device=REID_DEVICE
)

print("OSNet model:", REID_MODEL_NAME)
print("Device:", reid_model.device)
print("Preprocessing:", reid_model.preprocessing)

## Section 9 — Extract crop embeddings

Generate an embedding for each representative crop. We use a small batch size to keep GPU memory usage low.

In [ ]:
crop_paths = crop_manifest["crop_path"].tolist()

embedding_start = time.time()

crop_embeddings = reid_model.extract_embeddings_from_paths(
    crop_paths,
    batch_size=EMBEDDING_BATCH_SIZE,
    num_workers=0
)

crop_embeddings = np.asarray(crop_embeddings, dtype=np.float32)

if crop_embeddings.ndim != 2:
    raise ValueError(
        f"Expected a 2D embedding matrix; received {crop_embeddings.shape}"
    )

if crop_embeddings.shape[0] != len(crop_manifest):
    raise ValueError("Crop count and embedding count do not match.")

if not np.isfinite(crop_embeddings).all():
    raise ValueError("OSNet returned non-finite embedding values.")

embedding_seconds = time.time() - embedding_start

print("Crop embedding shape:", crop_embeddings.shape)
print("Embedding dimensions:", crop_embeddings.shape[1])
print("Embedding time (seconds):", round(embedding_seconds, 2))

## Section 10 — Aggregate embeddings into one feature per tracklet

First L2-normalize each crop embedding. Then average the embeddings belonging to the same `(video_id, track_id)` and normalize the average again.

The resulting vector represents that tracklet. This is a simple baseline; later we can compare it with quality-weighted or multi-prototype identity representations.

In [ ]:
def l2_normalize(vector, axis=-1, epsilon=1e-12):
    norm = np.linalg.norm(vector, axis=axis, keepdims=True)
    return vector / np.maximum(norm, epsilon)

normalized_crop_embeddings = l2_normalize(crop_embeddings)

tracklet_keys = sorted(
    crop_manifest["tracklet_key"].astype(str).unique().tolist()
)

tracklet_vectors = []
tracklet_metadata_rows = []

crop_manifest = crop_manifest.reset_index(drop=True)

for tracklet_key in tracklet_keys:
    indices = crop_manifest.index[
        crop_manifest["tracklet_key"] == tracklet_key
    ].to_numpy()

    mean_vector = normalized_crop_embeddings[indices].mean(axis=0)
    mean_vector = l2_normalize(mean_vector)

    tracklet_vectors.append(mean_vector)

    group = crop_manifest[
        crop_manifest["tracklet_key"] == tracklet_key
    ]

    # Use the full tracking log for tracklet time intervals, not only the
    # five selected crop frames. This makes same-video overlap checks safer.
    full_tracklet = track_log[
        track_log["tracklet_key"] == tracklet_key
    ]

    tracklet_metadata_rows.append({
        "tracklet_key": tracklet_key,
        "video_id": str(group["video_id"].iloc[0]),
        "track_id": int(group["track_id"].iloc[0]),
        "first_frame": int(full_tracklet["frame"].min()),
        "last_frame": int(full_tracklet["frame"].max()),
        "first_seen": float(full_tracklet["time_seconds"].min()),
        "last_seen": float(full_tracklet["time_seconds"].max()),
        "crop_samples": len(group),
        "mean_confidence": float(group["confidence"].mean()),
        "source_order": video_metadata[
            str(group["video_id"].iloc[0])
        ]["source_order"]
    })

tracklet_vectors = np.vstack(tracklet_vectors).astype(np.float32)
tracklet_summary = pd.DataFrame(tracklet_metadata_rows)

print("Tracklet feature matrix:", tracklet_vectors.shape)
print("Tracklets represented:", len(tracklet_summary))

display(tracklet_summary.head(20))

## Section 11 — Compare every compatible pair of tracklets

We compare tracklets using cosine similarity between normalized OSNet vectors.

Compatibility rules:
- Tracklets from different videos may be compared; their timestamps are treated as independent.
- Tracklets from the same video may only be compared if their observed frame intervals do not overlap.
- Local tracker IDs are never compared without their video IDs.

For each tracklet we calculate its best and second-best compatible match. The margin is:

`best_similarity - second_best_similarity`

A large margin means the best match is clearly separated from the next candidate. It is still not a probability.

In [ ]:
metadata_by_key = tracklet_summary.set_index("tracklet_key").to_dict("index")
vector_index = {
    key: index for index, key in enumerate(tracklet_keys)
}

pair_records = []
scores_by_tracklet = {key: [] for key in tracklet_keys}

for i, key_a in enumerate(tracklet_keys):
    meta_a = metadata_by_key[key_a]

    for j in range(i + 1, len(tracklet_keys)):
        key_b = tracklet_keys[j]
        meta_b = metadata_by_key[key_b]

        same_video = meta_a["video_id"] == meta_b["video_id"]

        if same_video:
            intervals_overlap = not (
                meta_a["last_frame"] < meta_b["first_frame"] or
                meta_b["last_frame"] < meta_a["first_frame"]
            )

            if intervals_overlap:
                continue

            if meta_a["last_frame"] < meta_b["first_frame"]:
                gap_frames = (
                    meta_b["first_frame"] - meta_a["last_frame"] - 1
                )
                earlier_key, later_key = key_a, key_b
            else:
                gap_frames = (
                    meta_a["first_frame"] - meta_b["last_frame"] - 1
                )
                earlier_key, later_key = key_b, key_a

            fps = video_metadata[meta_a["video_id"]]["fps"]
            gap_seconds = max(0, gap_frames) / fps
        else:
            # Separate videos have independent timelines.
            gap_frames = None
            gap_seconds = None
            earlier_key = None
            later_key = None

        similarity = float(
            tracklet_vectors[vector_index[key_a]] @
            tracklet_vectors[vector_index[key_b]]
        )

        pair_records.append({
            "tracklet_a": key_a,
            "tracklet_b": key_b,
            "video_a": meta_a["video_id"],
            "video_b": meta_b["video_id"],
            "same_video": same_video,
            "earlier_tracklet": earlier_key,
            "later_tracklet": later_key,
            "gap_frames": gap_frames,
            "gap_seconds": gap_seconds,
            "cosine_similarity": similarity
        })

        scores_by_tracklet[key_a].append((similarity, key_b))
        scores_by_tracklet[key_b].append((similarity, key_a))

candidate_df = pd.DataFrame(pair_records)

# Rank each tracklet's compatible candidates.
ranked_candidates = {}

for key, candidates in scores_by_tracklet.items():
    ranked_candidates[key] = sorted(
        candidates,
        key=lambda item: item[0],
        reverse=True
    )

best_candidate = {}
best_margin = {}

for key, ranked in ranked_candidates.items():
    if not ranked:
        best_candidate[key] = None
        best_margin[key] = np.nan
        continue

    best_score, best_key = ranked[0]
    best_candidate[key] = best_key

    if len(ranked) >= 2:
        second_score = ranked[1][0]
        best_margin[key] = best_score - second_score
    else:
        # There is no competing compatible candidate.
        best_margin[key] = np.nan

if not candidate_df.empty:
    rank_a = []
    rank_b = []
    margin_a = []
    margin_b = []
    mutual_best = []
    decision = []

    for _, row in candidate_df.iterrows():
        key_a = row["tracklet_a"]
        key_b = row["tracklet_b"]
        score = row["cosine_similarity"]

        ranked_a = ranked_candidates[key_a]
        ranked_b = ranked_candidates[key_b]

        rank_a_value = next(
            idx + 1 for idx, item in enumerate(ranked_a)
            if item[1] == key_b
        )
        rank_b_value = next(
            idx + 1 for idx, item in enumerate(ranked_b)
            if item[1] == key_a
        )

        margin_a_value = best_margin[key_a]
        margin_b_value = best_margin[key_b]

        is_mutual = (
            best_candidate[key_a] == key_b and
            best_candidate[key_b] == key_a
        )

        margin_a_ok = (
            np.isnan(margin_a_value) or
            margin_a_value >= MIN_SIMILARITY_MARGIN
        )
        margin_b_ok = (
            np.isnan(margin_b_value) or
            margin_b_value >= MIN_SIMILARITY_MARGIN
        )

        if (
            score >= AUTO_MATCH_THRESHOLD and
            is_mutual and margin_a_ok and margin_b_ok
        ):
            decision_value = "high_similarity_proposal"
        elif score >= REVIEW_THRESHOLD:
            decision_value = "review_required"
        else:
            decision_value = "low_similarity"

        rank_a.append(rank_a_value)
        rank_b.append(rank_b_value)
        margin_a.append(margin_a_value)
        margin_b.append(margin_b_value)
        mutual_best.append(is_mutual)
        decision.append(decision_value)

    candidate_df["rank_for_a"] = rank_a
    candidate_df["rank_for_b"] = rank_b
    candidate_df["margin_for_a"] = margin_a
    candidate_df["margin_for_b"] = margin_b
    candidate_df["mutual_best"] = mutual_best
    candidate_df["initial_decision"] = decision

    candidate_df = candidate_df.sort_values(
        "cosine_similarity",
        ascending=False
    ).reset_index(drop=True)

CANDIDATES_PATH = REID_OUTPUT_DIR / "multi_video_candidate_matches.csv"
candidate_df.to_csv(CANDIDATES_PATH, index=False)

print("Compatible tracklet pairs:", len(candidate_df))
print("Candidate file:", CANDIDATES_PATH)

display(
    candidate_df[
        [
            "tracklet_a", "tracklet_b", "cosine_similarity",
            "same_video", "gap_seconds", "rank_for_a",
            "rank_for_b", "margin_for_a", "margin_for_b",
            "initial_decision"
        ]
    ].head(30)
)

## Section 12 — Build provisional global identities

This is the global identity assignment step.

Only pairs that pass **all** of the following rules are used for an automatic provisional link:
1. Cosine similarity is at or above `AUTO_MATCH_THRESHOLD`.
2. Each tracklet selects the other as its best compatible match (mutual nearest neighbour).
3. Each best match has a sufficient margin over its second-best candidate, when a second candidate exists.
4. Merging the two groups does not place overlapping tracklets from the same video into one identity.

All other plausible pairs remain in the review queue. Each unmerged tracklet receives its own provisional global ID.

The IDs are anonymous and should be treated as provisional, not verified identities.

In [ ]:
# Union-find structure for conservative provisional clustering.
parent = {key: key for key in tracklet_keys}
cluster_members = {key: {key} for key in tracklet_keys}

def find_root(key):
    while parent[key] != key:
        parent[key] = parent[parent[key]]
        key = parent[key]
    return key

def clusters_can_merge(members_a, members_b):
    # For the same video, two tracklets representing one person cannot
    # be observed at overlapping frame intervals.
    for key_a in members_a:
        meta_a = metadata_by_key[key_a]

        for key_b in members_b:
            meta_b = metadata_by_key[key_b]

            if meta_a["video_id"] != meta_b["video_id"]:
                continue

            overlap = not (
                meta_a["last_frame"] < meta_b["first_frame"] or
                meta_b["last_frame"] < meta_a["first_frame"]
            )

            if overlap:
                return False

    return True

def merge_clusters(key_a, key_b):
    root_a = find_root(key_a)
    root_b = find_root(key_b)

    if root_a == root_b:
        return True

    members_a = cluster_members[root_a]
    members_b = cluster_members[root_b]

    if not clusters_can_merge(members_a, members_b):
        return False

    # Merge the smaller set into the larger set.
    if len(members_a) < len(members_b):
        root_a, root_b = root_b, root_a
        members_a, members_b = members_b, members_a

    parent[root_b] = root_a
    cluster_members[root_a] = members_a.union(members_b)
    del cluster_members[root_b]

    return True


# Apply only high-similarity proposals, from strongest to weakest.
merge_results = {}

if not candidate_df.empty:
    high_candidates = candidate_df[
        candidate_df["initial_decision"] == "high_similarity_proposal"
    ].sort_values("cosine_similarity", ascending=False)

    for _, row in high_candidates.iterrows():
        key_a = row["tracklet_a"]
        key_b = row["tracklet_b"]

        accepted = merge_clusters(key_a, key_b)

        merge_results[(key_a, key_b)] = (
            "provisional_link_created"
            if accepted else
            "rejected_same_video_overlap_conflict"
        )

# Create deterministic provisional Global IDs by source order and first frame.
ordered_keys = sorted(
    tracklet_keys,
    key=lambda key: (
        metadata_by_key[key]["source_order"],
        metadata_by_key[key]["first_frame"],
        key
    )
)

root_to_global_id = {}

for key in ordered_keys:
    root = find_root(key)

    if root not in root_to_global_id:
        root_to_global_id[root] = f"G{len(root_to_global_id) + 1:04d}"

global_id_by_key = {
    key: root_to_global_id[find_root(key)]
    for key in tracklet_keys
}

cluster_size_by_key = {
    key: len(cluster_members[find_root(key)])
    for key in tracklet_keys
}

# Update candidate decisions with actual cluster results.
if not candidate_df.empty:
    cluster_decisions = []

    for _, row in candidate_df.iterrows():
        pair_key = (row["tracklet_a"], row["tracklet_b"])

        if pair_key in merge_results:
            cluster_decisions.append(merge_results[pair_key])
        else:
            cluster_decisions.append("not_auto_linked")

    candidate_df["cluster_decision"] = cluster_decisions
    candidate_df.to_csv(CANDIDATES_PATH, index=False)

print("Tracklets:", len(tracklet_keys))
print("Provisional global identities:", len(set(global_id_by_key.values())))
print(
    "Tracklets included in multi-tracklet identities:",
    sum(size > 1 for size in cluster_size_by_key.values())
)

## Section 13 — Build the global identity registry

The registry has one row per tracklet. It records the video, local tracker ID, provisional global ID, observation range, number of crop samples and best candidate.

`identity_status` distinguishes:
- `provisional_high_similarity_link`: joined to another tracklet by the conservative matching rules.
- `review_required`: has a plausible candidate but was not automatically linked.
- `new_unmatched_tracklet`: no candidate reached the review threshold.
- `new_tracklet_no_compatible_candidate`: there was no temporally compatible candidate to compare.

The `global_id` for a review-required tracklet remains separate until the proposed match is accepted.

In [ ]:
registry_rows = []

for key in ordered_keys:
    meta = metadata_by_key[key]
    ranked = ranked_candidates[key]
    best_key = best_candidate[key]

    if best_key is None:
        best_score = np.nan
        suggested_global_id = ""
        status = "new_tracklet_no_compatible_candidate"
    else:
        best_score = float(ranked[0][0])
        suggested_global_id = global_id_by_key[best_key]

        if cluster_size_by_key[key] > 1:
            status = "provisional_high_similarity_link"
        elif best_score >= REVIEW_THRESHOLD:
            status = "review_required"
        else:
            status = "new_unmatched_tracklet"

    registry_rows.append({
        "global_id": global_id_by_key[key],
        "video_id": meta["video_id"],
        "track_id": int(meta["track_id"]),
        "tracklet_key": key,
        "first_seen_seconds": float(meta["first_seen"]),
        "last_seen_seconds": float(meta["last_seen"]),
        "first_frame": int(meta["first_frame"]),
        "last_frame": int(meta["last_frame"]),
        "crop_samples": int(meta["crop_samples"]),
        "best_candidate_tracklet": best_key or "",
        "best_candidate_global_id": suggested_global_id,
        "best_similarity": best_score,
        "match_status": status,
        "identity_verified": False
    })

global_registry = pd.DataFrame(registry_rows)

REGISTRY_PATH = REID_OUTPUT_DIR / "global_identity_registry.csv"
global_registry.to_csv(REGISTRY_PATH, index=False)

ASSIGNMENTS_PATH = REID_OUTPUT_DIR / "tracklet_global_id_assignments.csv"
global_registry[
    [
        "video_id", "track_id", "tracklet_key", "global_id",
        "best_candidate_tracklet", "best_candidate_global_id",
        "best_similarity", "match_status", "identity_verified"
    ]
].to_csv(ASSIGNMENTS_PATH, index=False)

print("Global identity registry:", REGISTRY_PATH)
print("Tracklet assignments:", ASSIGNMENTS_PATH)
print("Provisional global ID count:", global_registry["global_id"].nunique())

display(global_registry.head(30))

## Section 14 — Create a human-review queue

This file contains all pairs above the review threshold. It includes high-similarity proposals as well as ambiguous matches so that the first identity decisions can be evaluated.

You do not need to review every low-scoring pair. For the first experiment, review the strongest candidate pairs and a sample of moderate candidates.

Fill `manual_label` with:
- `same`
- `different`
- `uncertain`

The `manual_label` is ground-truth feedback for calibrating the model. It is not used to change the saved registry until we explicitly add a reviewed-merge step.

In [ ]:
if candidate_df.empty:
    review_queue = pd.DataFrame()
else:
    review_queue = candidate_df[
        candidate_df["cosine_similarity"] >= REVIEW_THRESHOLD
    ].copy()

    review_queue["manual_label"] = ""
    review_queue["review_notes"] = ""

REVIEW_QUEUE_PATH = REID_OUTPUT_DIR / "global_identity_manual_review.csv"
review_queue.to_csv(REVIEW_QUEUE_PATH, index=False)

print("Pairs in manual review queue:", len(review_queue))
print("Review queue saved:", REVIEW_QUEUE_PATH)

if not review_queue.empty:
    display(
        review_queue[
            [
                "tracklet_a", "tracklet_b", "video_a", "video_b",
                "cosine_similarity", "same_video",
                "gap_seconds", "initial_decision", "manual_label"
            ]
        ].head(50)
    )

## Section 15 — Visualize candidate pairs

This creates a side-by-side crop sheet for the highest-ranked candidates. The video ID and local tracker ID are displayed together so that you can identify the source of each crop.

For different videos, use the source video/timestamp when the crop itself is ambiguous. Do not assume a match merely because the clothing colours are similar.

In [ ]:
# Representative crop path for every tracklet.
representative_path_by_key = (
    crop_manifest
    .sort_values("quality_score", ascending=False)
    .drop_duplicates("tracklet_key")
    .set_index("tracklet_key")["crop_path"]
    .to_dict()
)

def read_crop_rgb(path):
    image = cv2.imread(str(path))
    if image is None:
        return None
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

NUMBER_OF_PAIRS_TO_DISPLAY = 12

if candidate_df.empty:
    print("No compatible candidate pairs to display.")
else:
    pairs_to_display = candidate_df.head(NUMBER_OF_PAIRS_TO_DISPLAY)

    fig, axes = plt.subplots(
        len(pairs_to_display), 2,
        figsize=(9, max(4, len(pairs_to_display) * 2.4))
    )

    if len(pairs_to_display) == 1:
        axes = np.array([axes])

    for row_index, (_, pair) in enumerate(pairs_to_display.iterrows()):
        key_a = pair["tracklet_a"]
        key_b = pair["tracklet_b"]

        image_a = read_crop_rgb(representative_path_by_key[key_a])
        image_b = read_crop_rgb(representative_path_by_key[key_b])

        axes[row_index, 0].axis("off")
        axes[row_index, 1].axis("off")

        if image_a is not None:
            axes[row_index, 0].imshow(image_a)
            axes[row_index, 0].set_title(key_a)

        if image_b is not None:
            axes[row_index, 1].imshow(image_b)
            axes[row_index, 1].set_title(key_b)

        axes[row_index, 0].set_ylabel(
            f"Similarity: {pair['cosine_similarity']:.3f}\n"
            f"Decision: {pair['initial_decision']}",
            rotation=0,
            labelpad=95,
            va="center"
        )

    plt.tight_layout()
    plt.show()

## Section 16 — Save the feature bank and experiment summary

The embedding bank stores one normalized feature vector per tracklet. The registry and assignment CSVs store how those tracklets were grouped.

For this first version, the registry is rebuilt as a batch from all videos listed in `VIDEO_SOURCES`. When adding another video, include the existing videos and the new video, then rerun the notebook over the complete set. Keep video IDs and source ordering consistent. The saved CSV/NPZ files preserve the last result, but global IDs are provisional and may be renumbered when the full batch is rebuilt. A truly incremental, immutable registry (for example, SQLite with locked IDs and reviewed merges) is a later engineering step.

The CSV registry is a saved, inspectable project artefact. It is not yet a transactional database; we can migrate it to SQLite after the matching rules have been evaluated.

In [ ]:
FEATURE_BANK_PATH = REID_OUTPUT_DIR / "global_tracklet_feature_bank.npz"
TRACKLET_SUMMARY_PATH = REID_OUTPUT_DIR / "global_tracklet_summary.csv"
EXPERIMENT_SUMMARY_PATH = REID_OUTPUT_DIR / "global_identity_experiment_summary.json"

np.savez_compressed(
    FEATURE_BANK_PATH,
    tracklet_keys=np.asarray(tracklet_keys, dtype=str),
    embeddings=tracklet_vectors.astype(np.float32)
)

tracklet_summary.to_csv(TRACKLET_SUMMARY_PATH, index=False)

summary = {
    "videos_processed": [str(source["video_id"]) for source in VIDEO_SOURCES],
    "tracklets_with_embeddings": int(len(tracklet_keys)),
    "compatible_pairs": int(len(candidate_df)),
    "high_similarity_proposals": (
        int((candidate_df["cluster_decision"] == "provisional_link_created").sum())
        if not candidate_df.empty and "cluster_decision" in candidate_df.columns
        else 0
    ),
    "review_queue_pairs": int(len(review_queue)),
    "provisional_global_ids": int(global_registry["global_id"].nunique()),
    "osnet_model": REID_MODEL_NAME,
    "auto_match_threshold": AUTO_MATCH_THRESHOLD,
    "review_threshold": REVIEW_THRESHOLD,
    "minimum_similarity_margin": MIN_SIMILARITY_MARGIN,
    "note": (
        "Global IDs are provisional. Cosine similarity is not a probability. "
        "High similarity proposals are not verified identities."
    )
}

with open(EXPERIMENT_SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print("Feature bank:", FEATURE_BANK_PATH)
print("Tracklet summary:", TRACKLET_SUMMARY_PATH)
print("Experiment summary:", EXPERIMENT_SUMMARY_PATH)
display(pd.DataFrame([summary]))

## Section 17 — How to add another video

For every additional video:

1. Give it a new, unique `video_id`, such as `session_02`.
2. Put its video file in `data/`.
3. Generate a detailed tracking log using Part 1/BoT-SORT or the optional helper in Section 3.
4. Ensure the tracking log belongs to that exact video and contains frame-level boxes.
5. Add the video and log paths to `VIDEO_SOURCES`.
6. Rerun from Section 1 so that all configured videos are compared together.

Example:

```python
VIDEO_SOURCES = [
    {
        "video_id": "session_01",
        "video_path": DATA_DIR / "drone_video.mov",
        "track_log_path": OUTPUT_DIR / "continuity_track_log.csv",
        "segment_start_seconds": None
    },
    {
        "video_id": "session_02",
        "video_path": DATA_DIR / "drone_video_02.mov",
        "track_log_path": OUTPUT_DIR / "drone_video_02_track_log.csv",
        "segment_start_seconds": 0.0
    }
]
```

Use videos that actually contain overlapping/repeated people if you want to test cross-video identity matching. If two recordings contain completely different people, no cross-video match should be expected.

For the first cross-video test, it is best to use two recordings from the same location or event with some of the same people visible in both, and to note which people you know are repeated. This gives us examples with which to evaluate matching accuracy.

## Section 18 — What to send back

Please share:

1. The crop gallery from Section 7.
2. The top 20 rows of `multi_video_candidate_matches.csv`.
3. `global_identity_registry.csv`.
4. `global_identity_manual_review.csv`.
5. The visual candidate sheet from Section 15.
6. How many videos were included and whether you know some people appear in more than one video.

## Part 4 conclusion

This notebook creates a first multi-video global identity matching layer. It uses OSNet appearance embeddings, mutual-best matching, a similarity margin, and a same-video overlap constraint to propose conservative links.

The registry keeps local tracker IDs separate from global IDs. The global IDs are anonymous and provisional. The next development step is to evaluate these proposals against manually known repeated people, calibrate thresholds, and then improve the identity store and counting logic.